# SonDatos × SUBROSA — Paso 3: Sentimiento (robertuito)
**Estudio Alofoke 2028**

Este notebook clasifica el sentimiento de los comentarios con GPU gratuita de Colab.

**Antes de correr:** menú `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU` → Guardar.

**Pasos:** ejecuta las celdas en orden (▶ o Shift+Enter). Al final descarga 2 archivos que llevas de vuelta a tu Mac.

In [1]:
# 1) Instalar pysentimiento (~2 min)
!pip install -q pysentimiento
print("Instalación completa ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.5 MB/s eta 0:00:00
Instalación completa ✓


In [2]:
# 2) Verificar GPU
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Sin GPU — activa T4 en Entorno de ejecución → Cambiar tipo. Funciona igual en CPU pero tarda ~30 min más.")

GPU disponible: True
GPU: Tesla T4


In [3]:
# 3) Subir comments_clean.parquet
# (está en tu Mac: sondatos-alofoke/data/processed/comments_clean.parquet)
from google.colab import files
uploaded = files.upload()
print("Archivo recibido ✓")

Saving comments_clean.parquet to comments_clean.parquet
Archivo recibido ✓


In [4]:
# 4) Cargar datos y filtrar utilizables
import pandas as pd

df = pd.read_parquet("comments_clean.parquet")
work = df[df["relevant"] & ~df["bot_flag"]].copy().reset_index(drop=True)
print(f"Comentarios a clasificar: {len(work):,}")
work["aspect"].value_counts()

Comentarios a clasificar: 39,618


,count
aspect,
general,26638
politico,10463
entretenimiento,2156
mixto,361


In [5]:
# 5) Clasificar sentimiento (con T4: ~5-10 min para ~40k comentarios)
from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="es")

BATCH = 256
labels, p_pos, p_neu, p_neg = [], [], [], []
texts = work["text_model"].fillna("").tolist()

for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    results = analyzer.predict(batch)
    for r in results:
        labels.append(r.output)
        p_pos.append(r.probas.get("POS", 0.0))
        p_neu.append(r.probas.get("NEU", 0.0))
        p_neg.append(r.probas.get("NEG", 0.0))
    if (i // BATCH) % 10 == 0:
        print(f"  {min(i+BATCH, len(texts)):,} / {len(texts):,}")

work["sentiment"] = labels
work["p_pos"] = p_pos
work["p_neu"] = p_neu
work["p_neg"] = p_neg
work["score"] = work["p_pos"] - work["p_neg"]
work["confidence"] = work[["p_pos", "p_neu", "p_neg"]].max(axis=1)

print("\nDistribución de sentimiento:")
print(work["sentiment"].value_counts(normalize=True).round(3))

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

...

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]


Distribución de sentimiento:
sentiment
NEG    0.428
NEU    0.407
POS    0.166
Name: proportion, dtype: float64


In [6]:
# 6) Guardar resultados + muestra de validación manual
work.to_parquet("comments_scored.parquet", index=False)

# muestra estratificada de 400 para etiquetado manual
n = 400
sample = (
    work.groupby(["sentiment", "aspect"], group_keys=False)
    .apply(lambda g: g.sample(min(len(g), max(1, int(n * len(g) / len(work)))), random_state=33))
    .head(n)
)
out = sample[["comment_id", "text", "sentiment", "aspect", "confidence"]].copy()
out["etiqueta_manual"] = ""
out["aspecto_manual"] = ""
out.to_csv("sample_to_label.csv", index=False, encoding="utf-8-sig")
print(f"Guardados: comments_scored.parquet ({len(work):,}) y sample_to_label.csv ({len(out)})")

Guardados: comments_scored.parquet (39,618) y sample_to_label.csv (396)


/tmp/ipykernel_652/1224253831.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), max(1, int(n * len(g) / len(work)))), random_state=33))


In [7]:
# 7) Descargar los 2 archivos a tu Mac
from google.colab import files
files.download("comments_scored.parquet")
files.download("sample_to_label.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Parte 2 · Análisis y gráficos

Continúa desde el paso 6 (usa el `comments_scored.parquet` que ya está en la sesión).
Si vuelves otro día con el archivo ya descargado, ejecuta primero la celda 8-bis para subirlo.

In [ ]:
# 8-bis) (Solo si empiezas sesión nueva) Subir comments_scored.parquet
import os
if not os.path.exists("comments_scored.parquet"):
    from google.colab import files
    files.upload()
import pandas as pd
work = pd.read_parquet("comments_scored.parquet")
print(f"Cargados: {len(work):,} comentarios clasificados")

In [ ]:
# 9) Configuración de eventos y marca (edita según el estudio)
EVENTOS = [
    {"fecha": "2025-03-05", "nombre": "Anuncio candidatura"},
    {"fecha": "2025-11-29", "nombre": "Rol de influencia"},
    {"fecha": "2026-06-21", "nombre": "Tuit viral"},
    {"fecha": "2026-06-24", "nombre": "PRSC abre puertas"},
    {"fecha": "2026-07-06", "nombre": "Presión de fans"},
]
SUJETO = "Alofoke"
TEMA = "claro"   # "claro" para documentos/PDF · "oscuro" para Instagram/web
MARCA = {
    "nombre": "SonDatos", "url": "sondatos.do",
    "oscuro": {"fondo":"#421034","pos":"#E1407A","neg":"#6B2D8B",
               "neutro":"#9A8496","acento":"#F3A0C0","texto":"#F5EDF2"},
    "claro":  {"fondo":"#FFFFFF","pos":"#D6336C","neg":"#6B2D8B",
               "neutro":"#8A7A85","acento":"#B03070","texto":"#2A1A24"},
}
AGREGACION = "W"          # W=semanal, D=diaria, ME=mensual
VENTANA_EVENTO_DIAS = 14
BOOTSTRAP_ITER = 2000
print("Configuración lista ✓")

In [ ]:
# 10) Análisis: serie semanal con IC bootstrap + impacto de eventos
import numpy as np
from scipy import stats
RNG = np.random.default_rng(33)

def net_ci(labels, n_boot=BOOTSTRAP_ITER):
    arr = labels.map({"POS":1,"NEU":0,"NEG":-1}).to_numpy()
    if len(arr) < 5: return (arr.mean() if len(arr) else np.nan), np.nan, np.nan
    boots = RNG.choice(arr, size=(n_boot, len(arr)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return arr.mean(), lo, hi

def serie(d):
    rows = []
    for w, g in d.groupby(pd.Grouper(key="published_at", freq=AGREGACION)):
        if g.empty: continue
        p, lo, hi = net_ci(g["sentiment"])
        rows.append({"week": w, "n": len(g), "net": p, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)

work["published_at"] = pd.to_datetime(work["published_at"], utc=True)
weekly = serie(work)
weekly.to_csv("serie_semanal.csv", index=False)
por_aspecto = {a: serie(g) for a, g in work.groupby("aspect")}

win = pd.Timedelta(days=VENTANA_EVENTO_DIAS)
imp = []
for ev in EVENTOS:
    d = pd.Timestamp(ev["fecha"], tz="UTC")
    pre  = work[(work["published_at"] >= d-win) & (work["published_at"] < d)]
    post = work[(work["published_at"] >= d) & (work["published_at"] < d+win)]
    row = {"evento": ev["nombre"], "fecha": ev["fecha"], "n_pre": len(pre), "n_post": len(post)}
    if len(pre) >= 20 and len(post) >= 20:
        a, b = (pre["sentiment"]=="NEG").sum(), (post["sentiment"]=="NEG").sum()
        pp = (a+b)/(len(pre)+len(post))
        se = np.sqrt(pp*(1-pp)*(1/len(pre)+1/len(post)))
        z = ((b/len(post))-(a/len(pre)))/se if se>0 else np.nan
        row.update({"neg_pre": a/len(pre), "neg_post": b/len(post),
                    "delta_pp": (b/len(post)-a/len(pre))*100,
                    "p_value": 2*(1-stats.norm.cdf(abs(z))),
                    "delta_net": net_ci(post["sentiment"],500)[0]-net_ci(pre["sentiment"],500)[0]})
    imp.append(row)
impacto = pd.DataFrame(imp)
impacto.to_csv("impacto_eventos.csv", index=False)

g, lo, hi = net_ci(work["sentiment"])
print(f"NET SENTIMENT GLOBAL: {g:+.3f} (IC95 [{lo:+.3f},{hi:+.3f}]) · n={len(work):,}")
for a, s in work.groupby("aspect"):
    p, l, h = net_ci(s["sentiment"])
    print(f"  {a:18s}: {p:+.3f} · n={len(s):,}")
print()
print(impacto.round(3).to_string(index=False))

In [ ]:
# 11) Gráficos con marca (mismos 4 del informe)
import matplotlib.pyplot as plt, matplotlib.dates as mdates
C = MARCA["oscuro" if TEMA == "oscuro" else "claro"]
FOOT = f"Fuente: comentarios públicos de YouTube · Modelo: robertuito · {MARCA['nombre']} — {MARCA['url']}"

def estilo(ax, fig):
    fig.patch.set_facecolor(C["fondo"]); ax.set_facecolor(C["fondo"])
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(C["neutro"])
    ax.tick_params(colors=C["texto"], labelsize=11)
    ax.grid(True, alpha=.22, color="#D8CDD4" if TEMA=="claro" else C["neutro"])
    ax.yaxis.label.set_color(C["texto"])

def eventos_v(ax):
    ym = ax.get_ylim()[1]
    for ev in EVENTOS:
        d = pd.Timestamp(ev["fecha"])
        ax.axvline(d, color=C["acento"], alpha=.55, lw=1, ls="--")
        ax.annotate(ev["nombre"], xy=(d, ym), fontsize=10, color=C["acento"],
                    rotation=90, va="top", ha="right")

def guardar(fig, nombre):
    fig.text(.5,.01,FOOT,ha="center",fontsize=7.5,color=C["neutro"])
    fig.savefig(f"{nombre}.png", dpi=300, bbox_inches="tight", facecolor=C["fondo"])
    plt.show(); plt.close(fig); print(f"  {nombre}.png ✓")

wk = weekly.copy(); wk["week"] = pd.to_datetime(wk["week"])

# --- 01 timeline ---
fig, ax = plt.subplots(figsize=(12,6)); estilo(ax,fig)
ax.fill_between(wk["week"], wk["lo"], wk["hi"], color=C["pos"], alpha=.13, label="IC 95%")
ax.plot(wk["week"], wk["net"], color=C["pos"], lw=2.8, label="Net sentiment (%POS − %NEG)")
ax.axhline(0, color=C["neutro"], lw=.8); eventos_v(ax)
ax.set_title(f"Sentimiento hacia {SUJETO} en la conversación política",
             color=C["texto"], fontsize=17, fontweight="bold", pad=18)
ax.set_ylabel("Net sentiment")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
leg = ax.legend(loc="lower left", frameon=False, fontsize=11)
[t.set_color(C["texto"]) for t in leg.get_texts()]
guardar(fig, "01_timeline")

# --- 02 aspectos ---
fig, ax = plt.subplots(figsize=(12,6)); estilo(ax,fig)
nombres = {"politico": "Como candidato", "entretenimiento": "Como comunicador"}
colores = {"politico": C["neg"], "entretenimiento": C["pos"]}
for a in ["politico", "entretenimiento"]:
    s = por_aspecto.get(a)
    if s is None or s.empty: continue
    s = s.copy(); s["week"] = pd.to_datetime(s["week"])
    ax.plot(s["week"], s["net"].rolling(2, min_periods=1).mean(),
            color=colores[a], lw=2.8, label=nombres[a])
ax.axhline(0, color=C["neutro"], lw=.8)
ax.set_title("¿Lo quieren como comunicador… y como candidato?",
             color=C["texto"], fontsize=17, fontweight="bold", pad=18)
ax.set_ylabel("Net sentiment")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
leg = ax.legend(loc="lower left", frameon=False, fontsize=11)
[t.set_color(C["texto"]) for t in leg.get_texts()]
guardar(fig, "02_aspectos")

# --- 03 eventos ---
ii = impacto.dropna(subset=["delta_net"]) if "delta_net" in impacto else pd.DataFrame()
if not ii.empty:
    fig, ax = plt.subplots(figsize=(11,5.5)); estilo(ax,fig)
    cols = [C["neg"] if d < 0 else C["pos"] for d in ii["delta_net"]]
    bars = ax.barh(ii["evento"], ii["delta_net"], color=cols)
    for b, (_, r) in zip(bars, ii.iterrows()):
        sig = " *" if r.get("p_value",1) < .05 else ""
        ax.text(b.get_width(), b.get_y()+b.get_height()/2, f' {r["delta_net"]:+.2f}{sig}',
                va="center", fontsize=9, color=C["texto"])
    ax.axvline(0, color=C["neutro"], lw=.8)
    ax.set_title("Cambio de sentimiento observado alrededor de cada evento",
                 color=C["texto"], fontsize=17, fontweight="bold", pad=18)
    ax.set_xlabel("Δ net sentiment (post − pre, ±14 días) · * p<0.05", color=C["texto"])
    guardar(fig, "03_eventos")

# --- 04 volumen ---
fig, ax = plt.subplots(figsize=(12,5)); estilo(ax,fig)
ax.bar(wk["week"], wk["n"], width=6, color=C["acento"], alpha=.9); eventos_v(ax)
ax.set_title("Volumen de conversación por semana",
             color=C["texto"], fontsize=17, fontweight="bold", pad=18)
ax.set_ylabel("Comentarios")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
guardar(fig, "04_volumen")

In [ ]:
# 12) Descargar gráficos y análisis (ZIP)
import zipfile
with zipfile.ZipFile("graficos_y_analisis.zip", "w") as z:
    for f in ["01_timeline.png","02_aspectos.png","03_eventos.png","04_volumen.png",
              "serie_semanal.csv","impacto_eventos.csv"]:
        try: z.write(f)
        except FileNotFoundError: pass
from google.colab import files
files.download("graficos_y_analisis.zip")
print("ZIP listo ✓")

## Notas
- **TEMA**: `"claro"` genera los gráficos para documentos (PDF/Word); `"oscuro"` la versión para Instagram y web con el fondo ciruela de marca.
- Los PNG salen a **300 dpi**, listos para impresión y para el informe.
- El título del gráfico 02 está pensado para el caso Alofoke — edítalo en la celda 11 para otros sujetos.
- Para el pipeline completo de un estudio nuevo (recolección incluida), usa `SonDatos_Pipeline_YouTube.ipynb`.